In [ ]:
!pip install -q timm huggingface_hub datasets

In [ ]:
import os

if not os.path.exists('CUB_200_2011'):
    !wget -q http://www.vision.caltech.edu/datasets/cub_200_2011/CUB_200_2011.tgz
    !tar -xzf CUB_200_2011.tgz
    print("Downloaded and extracted.")
else:
    print("Already present.")

DATA_ROOT = 'CUB_200_2011'

tar (child): CUB_200_2011.tgz: Cannot open: No such file or directory
tar (child): Error is not recoverable: exiting now
tar: Child returned status 2
tar: Error is not recoverable: exiting now
Downloaded and extracted.


In [ ]:
import os

if not os.path.exists('CUB_200_2011'):
    !wget -q https://data.caltech.edu/records/65de6-vp158/files/CUB_200_2011.tgz
    !tar -xzf CUB_200_2011.tgz
    print("Downloaded and extracted.")
else:
    print("Already present.")

DATA_ROOT = 'CUB_200_2011'


Downloaded and extracted.


In [ ]:
!ls -la CUB_200_2011.tgz
!file CUB_200_2011.tgz
!ls CUB_200_2011/ | head

-rw-r--r-- 1 root root 1150585339 Dec 11  2025 CUB_200_2011.tgz
CUB_200_2011.tgz: gzip compressed data, last modified: Thu Nov  3 19:00:23 2011, from Unix, original size modulo 2^32 1259479040 gzip compressed data, unknown method, has CRC, was "", encrypted, from FAT filesystem (MS-DOS, OS/2, NT), original size modulo 2^32 1259479040
attributes
bounding_boxes.txt
classes.txt
image_class_labels.txt
images
images.txt
parts
README
train_test_split.txt


In [ ]:
import pandas as pd

images = pd.read_csv(f'{DATA_ROOT}/images.txt', sep=' ', names=['img_id', 'filepath'])
labels = pd.read_csv(f'{DATA_ROOT}/image_class_labels.txt', sep=' ', names=['img_id', 'class_id'])
split = pd.read_csv(f'{DATA_ROOT}/train_test_split.txt', sep=' ', names=['img_id', 'is_train'])
classes = pd.read_csv(f'{DATA_ROOT}/classes.txt', sep=' ', names=['class_id', 'class_name'])

df = images.merge(labels, on='img_id').merge(split, on='img_id')
df['filepath'] = df['filepath'].apply(lambda x: f'{DATA_ROOT}/images/{x}')
df['class_id'] = df['class_id'] - 1  # zero-index

train_df = df[df['is_train'] == 1].reset_index(drop=True)
test_df = df[df['is_train'] == 0].reset_index(drop=True)

num_classes = classes.shape[0]
print(f"Train: {len(train_df)}, Test: {len(test_df)}, Classes: {num_classes}")

Train: 5994, Test: 5794, Classes: 200


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms

class BirdDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['filepath']).convert('RGB')
        img = self.transform(img)
        return img, row['class_id']

IMG_SIZE = 224
normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

# Baseline clean transform
clean_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    normalize
])

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.ToTensor(),
    normalize
])

# Simulated occlusion: aggressive random crop + random erasing
occlusion_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    normalize,
    transforms.RandomErasing(p=1.0, scale=(0.15, 0.35))
])

# Simulated low light: reduce brightness/contrast
lowlight_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ColorJitter(brightness=(0.25, 0.35), contrast=(0.5, 0.7)),
    transforms.ToTensor(),
    normalize
])

# Simulated background clutter/motion: blur + noise
clutter_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.GaussianBlur(kernel_size=5, sigma=(1.5, 3.0)),
    transforms.ToTensor(),
    normalize
])

train_ds = BirdDataset(train_df, train_tf)
test_clean_ds = BirdDataset(test_df, clean_tf)
test_occlusion_ds = BirdDataset(test_df, occlusion_tf)
test_lowlight_ds = BirdDataset(test_df, lowlight_tf)
test_clutter_ds = BirdDataset(test_df, clutter_tf)

BATCH = 32
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=2)
test_loaders = {
    'clean': DataLoader(test_clean_ds, batch_size=BATCH, num_workers=2),
    'occlusion': DataLoader(test_occlusion_ds, batch_size=BATCH, num_workers=2),
    'low_light': DataLoader(test_lowlight_ds, batch_size=BATCH, num_workers=2),
    'clutter_blur': DataLoader(test_clutter_ds, batch_size=BATCH, num_workers=2),
}

In [ ]:
import timm
import torch.nn as nn

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)

model = timm.create_model('resnet18', pretrained=True, num_classes=num_classes)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

Device: cuda


model.safetensors: reconstructing file:   0%|          |  0.00B / 46.8MB            

model.safetensors: downloading bytes:           |  0.00B            

In [ ]:
from tqdm import tqdm

EPOCHS = 10

for epoch in range(EPOCHS):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for imgs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * imgs.size(0)
        correct += (out.argmax(1) == labels).sum().item()
        total += imgs.size(0)

    scheduler.step()
    print(f"Epoch {epoch+1}: loss={total_loss/total:.4f}, train_acc={correct/total:.4f}")

torch.save(model.state_dict(), 'bird_resnet18.pth')
print("Saved model weights.")

Epoch 1/10: 100%|██████████| 188/188 [00:44<00:00,  4.25it/s]


Epoch 1: loss=5.0366, train_acc=0.0485


Epoch 2/10: 100%|██████████| 188/188 [00:42<00:00,  4.43it/s]


Epoch 2: loss=3.6211, train_acc=0.2871


Epoch 3/10: 100%|██████████| 188/188 [00:44<00:00,  4.24it/s]


...

Epoch 8: loss=0.9889, train_acc=0.8091


Epoch 9/10: 100%|██████████| 188/188 [00:43<00:00,  4.27it/s]


Epoch 9: loss=0.9252, train_acc=0.8272


Epoch 10/10: 100%|██████████| 188/188 [00:45<00:00,  4.13it/s]


Epoch 10: loss=0.9017, train_acc=0.8330
Saved model weights.


In [ ]:
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
import json

def evaluate(loader, name):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in tqdm(loader, desc=f"Eval: {name}"):
            imgs = imgs.to(device)
            out = model(imgs)
            preds = out.argmax(1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())

    acc = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average='macro', zero_division=0
    )
    return {'condition': name, 'accuracy': acc, 'precision': precision, 'recall': recall, 'f1_macro': f1}

results = []
for name, loader in test_loaders.items():
    results.append(evaluate(loader, name))

results_df = pd.DataFrame(results)
print(results_df)

results_df.to_csv('robustness_results.csv', index=False)
with open('robustness_results.json', 'w') as f:
    json.dump(results, f, indent=2)

Eval: clutter_blur: 100%|██████████| 182/182 [00:46<00:00,  3.94it/s]

      condition  accuracy  precision    recall  f1_macro
0         clean  0.677943   0.681757  0.680049  0.674594
1     occlusion  0.368485   0.494625  0.370641  0.392664
2     low_light  0.518985   0.593389  0.522102  0.526119
3  clutter_blur  0.443390   0.540627  0.449881  0.444633


In [ ]:
from huggingface_hub import notebook_login, HfApi, create_repo

notebook_login()  # paste your free HF token when prompted

HF_USERNAME = "dhayal1"   # replace this
HF_REPO = f"{HF_USERNAME}/bird-vision-robustness"

create_repo(HF_REPO, exist_ok=True)

api = HfApi()
api.upload_file(
    path_or_fileobj="bird_resnet18.pth",
    path_in_repo="bird_resnet18.pth",
    repo_id=HF_REPO
)
api.upload_file(
    path_or_fileobj="robustness_results.json",
    path_in_repo="robustness_results.json",
    repo_id=HF_REPO
)

# Also save the class names so the deployed app can map predictions to species
classes.to_csv('classes.csv', index=False)
api.upload_file(
    path_or_fileobj="classes.csv",
    path_in_repo="classes.csv",
    repo_id=HF_REPO
)

print(f"Uploaded to https://huggingface.co/{HF_REPO}")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  /content/bird_resnet18.pth  :   1%|1         |  561kB / 45.2MB            

Uploaded to https://huggingface.co/dhayal1/bird-vision-robustness
